# 04 - Diagnostics and calibration

Everything about a saved run: its settings, its training record (health tiles and every per-epoch metric), the
test block re-scored with the baselines' noise tests, every head analysed per horizon, the learned indicator
periods, and an interactive **calibration explorer** that refits the calibration pipeline on the calibration
block with another interval scale, with or without delta shrinkage, at another miscoverage level. Nothing here
writes to the run.

In [1]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"

In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

from neural_trade.evaluation.baselines import BaselineSet
from neural_trade.evaluation.report import evaluate
from neural_trade.notebook import CalibrationExplorer, load_run_blocks, pick_run
from neural_trade.registries.visualizations import Visualizations
from neural_trade.visualization import analytics_tables as AT
from neural_trade.visualization.indicator_evolution import applied_periods, indicator_applied_periods, indicator_summary
from neural_trade.visualization.training_dashboard import training_health_html

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
blocks = load_run_blocks(run_dir, csv_path=CSV_PATH)
test, cfg = blocks["test"], blocks["config"]
metrics = run_dir / "metrics.jsonl"
display(AT.styled(AT.run_settings_table(run_dir, cfg)))

run: ..\runs\20260924T165923Z-fa75177-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T165923Z-fa75177-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


## Training record

In [3]:
display(HTML(training_health_html(metrics, cfg)))
Visualizations.build("training_dashboard", metrics, cfg).show()
Visualizations.build("training_direction_detail", metrics, cfg).show()
Visualizations.build("training_loss_terms", metrics, cfg).show()

term,weight in the total,epoch 18 (served): val,share of val,train,epoch 20 (last): val
point,λ 1.86 / 0.717 / 0.684 (inside),0.4533,9.2%,0.9687,0.4522
trend,λ_ext 0.1 (inside) × 1,0.03795,0.8%,0.09757,0.03864
direction BCE,λ_dir 0.659 × 1,1.387,28.0%,1.35,1.392
NLL,λ_var 0.369 × 1,0.9637,19.5%,1.452,0.9672
CRPS,λ_crps 0.775,0.7669,15.5%,1.182,0.768
soft ECE,λ_soft_ece 2.24,0.824,16.6%,0.4349,0.8486
volatility,λ_vol 3.32 (inside) × 0.1,0.1262,2.5%,0.1653,0.1293
physics,λ 0.1 (inside),0.2179,4.4%,0.1196,0.2376
T-perp,,0.1885,,0.01818,0.2045
Casimir,,0.000128,,4.387e-05,7.298e-05


## The TEST block, re-scored

Baselines refit on the train block, so the noise tests appear even for runs saved before they existed. Served heads and raw price heads side by side.

In [4]:
pipe = blocks["predictor"].bundle.calibration_pipeline
betas = pipe.delta_scale if pipe is not None else None
raw = blocks["test_raw"].delta
tr = blocks["blocks"]["train"]
report = evaluate(test, cfg, baselines=BaselineSet.fit(tr["X"], tr["y"], tr["last_close"], cfg.DIR_DEADBAND_BPS),
                  cal_frame=blocks["cal"], run_id=run_dir.name, raw_delta=raw, delta_scale=betas)
display(Markdown(report.to_markdown()))
display(AT.styled(AT.baseline_table(report)))
display(AT.styled(AT.classification_table(test, cfg)))
display(AT.styled(AT.delta_quality_table(test, cfg, raw_delta=raw, delta_scale=betas)))

# Evaluation report - test split - run `20260924T165923Z-fa75177-dirty-af67ee43`

n = 7236 samples, one per 1-minute bar. Direction metrics count only moves beyond 5 bps (the neutral mask). Consecutive samples share most of their target window, so n_eff = n // bars ahead counts the non-overlapping outcomes.

## Direction heads: P(up) > 0.5 on moves beyond 5 bps

Up is the positive class. Temperature scaling does not move P(up) across 0.5, so the counts and rates are the same before and after calibration; Brier and ECE are not.

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| n scored (outside the deadband) | 5345 | 5697 | 5874 |
| n_eff of the scored moves (n scored // bars ahead) | 534 | 379 | 293 |
| true up-rate | 0.5139 | 0.5147 | 0.5189 |
| calls up (predicted up-rate) | 0.3908 | 0.5431 | 0.4025 |
| accuracy | 0.5158 | 0.4941 | 0.5230 |
| balanced accuracy | 0.5189 | 0.4929 | 0.5267 |
| precision (up) | 0.5381 | 0.5081 | 0.5520 |
| recall / sensitivity (up) | 0.4092 | 0.5362 | 0.4281 |
| specificity (down) | 0.6286 | 0.4495 | 0.6253 |
| F1 (up) | 0.4648 | 0.5217 | 0.4823 |
| MCC | 0.0387 | -0.0143 | 0.0544 |
| AUC | 0.5245 | 0.4948 | 0.5288 |
| Brier | 0.2502 | 0.2520 | 0.2501 |
| ECE (positive class) | 0.0239 | 0.0349 | 0.0277 |
| ECE of a constant 0.5 (= distance of the up-rate from 0.5) | 0.0139 | 0.0147 | 0.0189 |
| TP / FP / TN / FN | 1124 / 965 / 1633 / 1623 | 1572 / 1522 / 1243 / 1360 | 1305 / 1059 / 1767 / 1743 |
| Gaussian readout: calls up | 0.0000 | 0.0000 | 0.0000 |
| Gaussian readout: MCC | 0.0000 | 0.0000 | 0.0000 |
| Gaussian readout: AUC | 0.5000 | 0.5000 | 0.5000 |
| Gaussian readout: Brier | 0.2500 | 0.2500 | 0.2500 |
| Gaussian readout: ECE | 0.0139 | 0.0147 | 0.0189 |

## Price heads (dollars)

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| RMSE ($), served | 196.21 | 236.10 | 269.11 |
| RMSE ($), raw heads | 197.73 | 247.04 | 279.07 |
| RMSE ($), zero prediction | 196.21 | 236.10 | 269.11 |
| MAE ($), served | 145.66 | 175.66 | 199.93 |
| MAE ($), raw heads | 146.68 | 183.63 | 208.19 |
| MAE ($), zero prediction | 145.66 | 175.66 | 199.93 |
| skill vs zero (1 - MSE / MSE of 0), served | 0.0000 | 0.0000 | 0.0000 |
| skill vs zero, raw heads | -0.0155 | -0.0948 | -0.0754 |
| EV, served | 0.0000 | 0.0000 | 0.0000 |
| EV, raw heads | -0.0126 | -0.0880 | -0.0696 |
| corr, Pearson (the same raw and served) | 0.0000 | 0.0000 | 0.0000 |
| corr, Spearman | 0.0000 | 0.0000 | 0.0000 |
| mean predicted ($), served | 0.00 | 0.00 | 0.00 |
| mean predicted ($), raw heads | -6.08 | -12.44 | -11.80 |
| mean realised ($) | 6.30 | 9.37 | 12.34 |
| share predicted up, raw heads | 0.3572 | 0.3393 | 0.4288 |
| share realised up | 0.5135 | 0.5146 | 0.5156 |
| shrink beta (served = beta x raw, fit on cal) | 0.0000 | 0.0000 | 0.0000 |

## Variance heads

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| CRPS ($) | 105.62 | 127.34 | 145.09 |
| CRPSS vs constant variance | 0.0209 | 0.0166 | 0.0187 |
| NLL | 6.6535 | 6.8456 | 6.9704 |
| PIT KS | 0.0337 | 0.0284 | 0.0364 |
| var / err^2 Spearman | 0.2489 | 0.2332 | 0.2213 |
| coverage of the 90% interval | 0.9027 | 0.9059 | 0.9124 |
| width of the 90% interval ($) | 643.12 | 781.37 | 911.88 |

## Confidence gap (accuracy of the more confident half minus the less confident half)

| horizon | gap | 95% CI | verdict |
|---|---|---|---|
| h0 | 0.0115 | [-0.0253, 0.0511] | NOISE |
| h1 | 0.0060 | [-0.0336, 0.0439] | NOISE |
| h2 | 0.0131 | [-0.0306, 0.0510] | NOISE |

## Coherence across horizons

Magnitude ordering: the share of samples whose predicted move grows with the horizon, as the loss asks. Magnitudes in random order would give 0.5, 0.5 and 0.1667.

| check | raw price heads (the trained ordering) | served deltas (beta-shrunk: h0 0.000 / h1 0.000 / h2 0.000) | realised moves |
|---|---|---|---|
| abs(d h0) <= abs(d h1) | 0.8347 | 1.0000 | 0.6039 |
| abs(d h1) <= abs(d h2) | 0.5969 | 1.0000 | 0.5712 |
| full chain h0 <= h1 <= h2 | 0.4608 | 1.0000 | 0.3159 |

beta = 0 for h0, h1, h2: the served delta is 0 there, so every sign and magnitude check on the served deltas is empty; the sign checks below use the raw heads.

The served ordering mostly reflects the per-horizon shrink beta, not the model: judge the trained constraint on the raw heads.

Sign agreement: sign(delta) against calibrated P(up) > 0.5 (the same for raw and served deltas while beta > 0).

| | h0 | h1 | h2 | all 3 |
|---|---|---|---|---|
| agree | 0.6054 | 0.5738 | 0.5949 | 0.2030 |
| expected if the two signs were independent | 0.5324 | 0.4849 | 0.5144 | 0.1324 |

- P(up) unanimity (all three horizons call the same side): 0.2692

## Against baselines (fit on the training block)

Each cell: model vs baseline, the margin (positive = the model is better) and the verdict. "noise": |z| < 1.96, so the ordering is not established; "significantly worse": the model loses with z <= -1.96. "DM z": Diebold-Mariano test of the per-sample loss difference (RMSE and skill, MAE, Brier, accuracy, CRPS, NLL) with a Bartlett (Newey-West) long-run variance, lag 2 x bars ahead. "boot z": the margin over its standard error in a paired moving-block bootstrap (80-bar blocks, 500 resamples; MCC, AUC, balanced accuracy, ECE, EV, corr, PIT KS, var / err^2 Spearman). Rows that only restate the RMSE verdict for a constant prediction (EV, corr and skill against zero_delta / mean_delta) are left out; the JSON keeps every verdict.

| baseline | metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|---|
| logreg_lags | direction/mcc | 0.0387 vs 0.0390 (-0.0003): does not beat, noise (boot z -0.01) | -0.0143 vs 0.0397 (-0.0540): does not beat, noise (boot z -1.58) | 0.0544 vs 0.0411 (+0.0133): beats, noise (boot z +0.33) |
| logreg_lags | direction/auc | 0.5245 vs 0.5241 (+0.0004): beats, noise (boot z +0.02) | 0.4948 vs 0.5228 (-0.0280): does not beat, noise (boot z -1.34) | 0.5288 vs 0.5319 (-0.0031): does not beat, noise (boot z -0.12) |
| logreg_lags | direction/brier | 0.2502 vs 0.2495 (-0.0007): does not beat, noise (DM z -0.54) | 0.2520 vs 0.2493 (-0.0027): does not beat, significantly worse (DM z -1.98) | 0.2501 vs 0.2492 (-0.0009): does not beat, noise (DM z -0.66) |
| logreg_lags | direction/ece_pos | 0.0239 vs 0.0211 (-0.0028): does not beat, noise (boot z -0.36) | 0.0349 vs 0.0197 (-0.0152): does not beat, noise (boot z -1.02) | 0.0277 vs 0.0231 (-0.0046): does not beat, noise (boot z -0.59) |
| logreg_lags | direction/acc | 0.5158 vs 0.5160 (-0.0002): does not beat, noise (DM z -0.01) | 0.4941 vs 0.5166 (-0.0225): does not beat, noise (DM z -1.28) | 0.5230 vs 0.5157 (+0.0073): beats, noise (DM z +0.38) |
| logreg_lags | direction/bal_acc | 0.5189 vs 0.5190 (-0.0002): does not beat, noise (boot z -0.01) | 0.4929 vs 0.5195 (-0.0266): does not beat, noise (boot z -1.58) | 0.5267 vs 0.5200 (+0.0067): beats, noise (boot z +0.34) |
| class_prior | direction/mcc | 0.0387 vs 0.0000 (+0.0387): beats, noise (boot z +1.60) | -0.0143 vs 0.0000 (-0.0143): does not beat, noise (boot z -0.59) | 0.0544 vs 0.0000 (+0.0544): beats (boot z +2.08) |
| class_prior | direction/auc | 0.5245 vs 0.5000 (+0.0245): beats, noise (boot z +1.54) | 0.4948 vs 0.5000 (-0.0052): does not beat, noise (boot z -0.33) | 0.5288 vs 0.5000 (+0.0288): beats, noise (boot z +1.66) |
| class_prior | direction/brier | 0.2502 vs 0.2502 (+0.0000): beats, noise (DM z +0.01) | 0.2520 vs 0.2502 (-0.0018): does not beat, noise (DM z -1.53) | 0.2501 vs 0.2502 (+0.0001): beats, noise (DM z +0.08) |
| class_prior | direction/ece_pos | 0.0239 vs 0.0208 (-0.0031): does not beat, noise (boot z -0.49) | 0.0349 vs 0.0196 (-0.0153): does not beat, noise (boot z -0.90) | 0.0277 vs 0.0234 (-0.0043): does not beat, noise (boot z -0.85) |
| class_prior | direction/acc | 0.5158 vs 0.4861 (+0.0297): beats, noise (DM z +1.62) | 0.4941 vs 0.4853 (+0.0088): beats, noise (DM z +0.34) | 0.5230 vs 0.4811 (+0.0419): beats, noise (DM z +1.85) |
| class_prior | direction/bal_acc | 0.5189 vs 0.5000 (+0.0189): beats, noise (boot z +1.59) | 0.4929 vs 0.5000 (-0.0071): does not beat, noise (boot z -0.59) | 0.5267 vs 0.5000 (+0.0267): beats (boot z +2.08) |
| zero_delta | delta/rmse | 196.21 vs 196.21 (+0.00, +0.00%): does not beat | 236.10 vs 236.10 (+0.00, +0.00%): does not beat | 269.11 vs 269.11 (+0.00, +0.00%): does not beat |
| zero_delta | delta/mae | 145.66 vs 145.66 (+0.00, +0.00%): does not beat | 175.66 vs 175.66 (+0.00, +0.00%): does not beat | 199.93 vs 199.93 (+0.00, +0.00%): does not beat |
| mean_delta | delta/rmse | 196.21 vs 196.24 (+0.02, +0.01%): beats, noise (DM z +1.05) | 236.10 vs 236.14 (+0.04, +0.02%): beats, noise (DM z +1.09) | 269.11 vs 269.17 (+0.06, +0.02%): beats, noise (DM z +1.12) |
| mean_delta | delta/mae | 145.66 vs 145.68 (+0.02, +0.01%): beats, noise (DM z +1.01) | 175.66 vs 175.69 (+0.03, +0.02%): beats, noise (DM z +0.94) | 199.93 vs 199.97 (+0.05, +0.02%): beats, noise (DM z +0.91) |
| const_var | variance/crps | 105.62 vs 107.87 (+2.25, +2.09%): beats (DM z +5.80) | 127.34 vs 129.50 (+2.16, +1.66%): beats (DM z +3.59) | 145.09 vs 147.86 (+2.77, +1.87%): beats (DM z +4.08) |
| const_var | variance/nll | 6.6535 vs 6.7057 (+0.0522): beats (DM z +3.55) | 6.8456 vs 6.8904 (+0.0447): beats (DM z +2.13) | 6.9704 vs 7.0219 (+0.0515): beats (DM z +2.70) |
| const_var | variance/pit_ks | 0.0337 vs 0.0663 (+0.0326): beats (boot z +5.99) | 0.0284 vs 0.0614 (+0.0330): beats (boot z +4.82) | 0.0364 vs 0.0644 (+0.0280): beats (boot z +4.58) |
| const_var | variance/corr_var_err2_spearman | 0.2489 vs 0.0000 (+0.2489): beats (boot z +6.99) | 0.2332 vs 0.0000 (+0.2332): beats (boot z +5.75) | 0.2213 vs 0.0000 (+0.2213): beats (boot z +5.29) |


horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,7236,7236,7236
n scored (moves beyond 5 bps),5345,5697,5874
n_eff of the scored moves (n scored // bars ahead),534,379,293
true up-rate,0.5139,0.5147,0.5189
calls up (predicted up-rate),0.3908,0.5431,0.4025
accuracy,0.5158,0.4941,0.5230
accuracy 95% CI (n_eff),"[0.4735, 0.5579]","[0.4441, 0.5442]","[0.4659, 0.5794]"
majority-class accuracy (hindsight),0.5139,0.5147,0.5189


horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,7236,7236,7236
n_eff (non-overlapping outcomes),723,482,361
shrink beta (served = beta x raw),0.0000,0.0000,0.0000
"RMSE ($), raw heads",197.73,247.04,279.07
"RMSE ($), served",196.21,236.10,269.11
"RMSE ($), zero prediction",196.21,236.10,269.11
"MAE ($), raw heads",146.68,183.63,208.19
"MAE ($), served",145.66,175.66,199.93


## The heads on the TEST block

In [5]:
Visualizations.build("direction_analytics", test, cfg).show()
Visualizations.build("delta_analytics", test, cfg, raw_delta=raw).show()
Visualizations.build("variance_analytics", test, cfg, raw_delta=raw).show()
Visualizations.build("confidence_analytics", test, cfg, var_scale=blocks["predictor"].bundle.meta.get("var_scale"),
                     report=report).show()
Visualizations.build("coherence_analytics", test, cfg, raw_delta=blocks["test_raw"]).show()

In [6]:
display(AT.styled(AT.magnitude_ordering_table(test, raw_delta=raw)))
display(AT.styled(AT.alignment_table(test, cfg, raw_delta=raw)))
display(AT.styled(AT.trailing_move_table({"cal": blocks["cal_raw"], "test": blocks["test_raw"]}, cfg)))

,raw heads (trained ordering),raw heads 95% CI (n_eff),served deltas,served deltas 95% CI (n_eff),realised moves,realised moves 95% CI (n_eff),magnitudes in random order,n_eff
check,,,,,,,,
|d h0| <= |d h1|,0.8347,"[0.7990, 0.8652]",1.0000,"[0.9921, 1.0000]",0.6039,"[0.5596, 0.6466]",0.5000,482
|d h1| <= |d h2|,0.5969,"[0.5456, 0.6461]",1.0000,"[0.9895, 1.0000]",0.5712,"[0.5197, 0.6212]",0.5000,361
full chain |d h0| <= |d h1| <= |d h2|,0.4608,"[0.4101, 0.5123]",1.0000,"[0.9895, 1.0000]",0.3159,"[0.2702, 0.3655]",0.1667,361


,agree,95% CI low,95% CI high,expected if independent,share delta > 0,share P(up) > 0.5,n,n_eff
horizon,,,,,,,,
h0,0.6054,0.5694,0.6404,0.5324,0.3572,0.3864,7236,723
h1,0.5738,0.5293,0.6172,0.4849,0.3393,0.5471,7236,482
h2,0.5949,0.5436,0.6443,0.5144,0.4288,0.3988,7236,361
all 3,0.2030,0.1648,0.2475,0.1324,n/a,n/a,7236,361


## Learned indicator periods

Base periods per epoch; the diamonds are the median applied (per-window shifted) periods of the served weights on the test windows.

In [7]:
applied = applied_periods(blocks["predictor"], test.X_raw, block="test")
Visualizations.build("indicator_evolution", metrics, cfg, applied=applied).show()
indicator_applied_periods(applied, cfg, metrics=metrics).show()
AT.styled(indicator_summary(metrics, cfg, applied=applied))

,indicator,start,after epoch 1,last,min,max,change %,slope %/epoch (last 5),headroom %,near bound,CV,corr with val loss,r with epoch,r of changes,served base,applied p5,applied p50,applied p95,applied p50 vs base %
macd_1_slow,MACD #1 slow,35.0000,37.1261,56.7346,35.0000,58.1518,62.0989,-0.7022,5.7556,,0.1532,-0.9611,0.9282,-0.2125,57.1899,52.2471,70.7204,90.8814,23.6589
macd_0_fast,MACD #0 fast,12.0000,11.0109,6.1148,5.8328,12.0000,-49.0432,-1.5783,205.7406,,0.1903,0.9022,-0.9152,0.1484,5.8328,3.9867,4.8261,6.5214,-17.2585
macd_2_fast,MACD #2 fast,8.0000,8.3611,4.4535,4.3583,8.4355,-44.3316,-2.9563,122.6737,,0.2083,0.9406,-0.9660,0.2380,4.3583,3.3495,3.9340,4.4392,-9.7371
bb_period_1,BB #1,20.0000,19.0057,27.7640,19.0057,27.9356,38.8200,1.5077,116.1073,,0.1149,-0.8702,0.9500,-0.1637,27.6961,21.9951,31.1774,42.9055,12.5695
macd_1_fast,MACD #1 fast,5.0000,5.0379,3.1708,3.1653,5.0379,-36.5831,-0.7482,58.5422,,0.1607,0.9659,-0.9788,0.5590,3.2231,2.3693,2.8112,3.8368,-12.7790
ma_period_2,MA #2,30.0000,26.8004,21.3456,21.0653,30.0000,-28.8481,-1.0732,181.0889,,0.0722,0.7464,-0.8941,-0.0465,21.5297,19.6423,25.4783,33.3440,18.3401
ma_period_0,MA #0,5.0000,4.8540,3.7643,3.7326,5.0000,-24.7139,-0.7582,88.2154,,0.0870,0.9548,-0.9692,0.5200,3.7326,2.9871,3.6762,4.6246,-1.5114
macd_0_signal,MACD #0 signal,9.0000,8.7108,6.9110,6.3492,9.0000,-23.2110,0.6007,245.5503,,0.0954,0.9325,-0.8861,0.6097,6.3881,6.2803,6.6690,7.7656,4.3975
macd_2_signal,MACD #2 signal,9.0000,8.5293,7.0146,6.4649,9.0000,-22.0600,-0.8333,250.7302,,0.0803,0.9532,-0.9174,0.7321,6.4649,6.6894,7.2930,8.5150,12.8103
macd_0_slow,MACD #0 slow,26.0000,27.2697,31.4014,26.0000,32.0308,20.7745,0.8325,91.0744,,0.0366,-0.3883,0.4427,0.7052,29.3139,19.5428,23.1956,26.9211,-20.8716


## Calibration explorer

Fit on the calibration block, scored on the test block; the run itself is not changed.

In [8]:
calib = CalibrationExplorer(blocks)
display(calib.widget())
calib.click_refit()   # the run's saved settings first; then change them and press Refit

Static copy of that refit (the explorer above stays interactive): the refit next to the run's saved pipeline, per horizon.

In [9]:
display(calib.comparison_table().round(4))
reliability, coverage = calib.figures("h1")
reliability.show()
coverage.show()

temperature  delta beta  coverage  target  mean width $  \
horizon pipeline                                                            
h0      refit          1.4863         0.0    0.9027     0.9      643.1248   
        saved          1.4863         0.0    0.9027     0.9      643.1248   
h1      refit          1.8209         0.0    0.9059     0.9      781.3664   
        saved          1.8209         0.0    0.9059     0.9      781.3664   
h2      refit          1.6038         0.0    0.9124     0.9      911.8846   
        saved          1.6038         0.0    0.9124     0.9      911.8846   

                  EV raw delta  EV served delta  ECE raw  ECE calibrated  \
horizon pipeline                                                           
h0      refit          -0.0126              0.0   0.0289          0.0239   
        saved          -0.0126              0.0   0.0289          0.0239   
h1      refit          -0.0880              0.0   0.0572          0.0349   
        saved          -0.0880              0.0   0.0572          0.0349   
h2      refit          -0.0696              0.0   0.0339          0.0277   
        saved          -0.0696              0.0   0.0339          0.0277   

                  coverage cal (in-sample)  ECE calibrated cal (in-sample)  \
horizon pipeline                                                             
h0      refit                       0.9006                          0.0705   
        saved                       0.9006                          0.0705   
h1      refit                       0.9006                          0.0876   
        saved                       0.9006                          0.0876   
h2      refit                       0.9006                          0.0711   
        saved                       0.9006                          0.0711   

                  up-rate cal  up-rate test  
horizon pipeline                             
h0      refit          0.4482        0.5139  
        saved          0.4482        0.5139  
h1      refit          0.4388        0.5147  
        saved          0.4388        0.5147  
h2      refit          0.4483        0.5189  
        saved          0.4483        0.5189